# AgriTech AI -- Exploratory Data Analysis & Model Training

**Final Project -- DS & ML Bootcamp (Feb 2026)**  
**Author:** Yasir  
**Dataset:** Kaggle Crop Recommendation Dataset (2,201 samples)

---

## Table of Contents
1. **Data Loading & Overview**
2. **Exploratory Data Analysis (EDA)**
3. **Data Preprocessing**
4. **Model Training & Comparison**
5. **Model Evaluation & Metrics**
6. **Feature Importance Analysis**
7. **Conclusions**

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully.')

## 2. Data Loading & Overview

We use the **Crop Recommendation Dataset** from Kaggle.  
It contains soil nutrient levels (N, P, K), weather conditions, and the recommended crop.

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/Crop_recommendation.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Total Samples: {df.shape[0]:,}')
print(f'Total Features: {df.shape[1] - 1}')
print(f'Target Column: "label"')
print(f'Unique Crops: {df["label"].nunique()}')
print(f'\nCrop Types: {sorted(df["label"].unique())}')

In [ ]:
# First 10 rows
df.head(10)

In [ ]:
# Dataset Info
df.info()

In [ ]:
# Statistical Summary
df.describe().round(2)

In [ ]:
# Check for missing values and duplicates
print('Missing Values per Column:')
print(df.isnull().sum())
print(f'\nTotal Missing: {df.isnull().sum().sum()}')
print(f'Duplicate Rows: {df.duplicated().sum()}')

> The dataset has **zero missing values** and **no duplicates**.  
> This means we can skip imputation and go straight to encoding and scaling.

## 3. Exploratory Data Analysis (EDA)

Let's visualize the data to discover patterns, distributions, and relationships.

### 3.1 Crop Distribution
How many samples do we have for each crop?

In [ ]:
# Crop Distribution
plt.figure(figsize=(14, 6))
crop_counts = df['label'].value_counts()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(crop_counts)))

bars = plt.bar(crop_counts.index, crop_counts.values, color=colors, edgecolor='white')
plt.title('Distribution of Crops in Dataset', fontsize=16, fontweight='bold')
plt.xlabel('Crop Type', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, crop_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             str(count), ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nEach crop has exactly {crop_counts.values[0]} samples -- perfectly balanced dataset.')

### 3.2 Feature Distributions
Let's see how each feature is distributed.

In [ ]:
# Feature Distributions
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.histplot(df[feature], kde=True, ax=axes[i], color=colors[i*3], edgecolor='white')
    axes[i].set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')

axes[-1].axis('off')
fig.suptitle('Distribution of All Features', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Correlation Heatmap
Let's check how the features correlate with each other.

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df[features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=1, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey Observations:')
print('  - Most features show low correlation (good for ML models)')
print('  - No severe multicollinearity issues detected')

### 3.4 Feature Variation Across Crops
Let's examine how features differ between crop types.

In [ ]:
# Box plots per feature across crops
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

top_crops = ['rice', 'maize', 'apple', 'coffee', 'cotton', 'orange']
subset = df[df['label'].isin(top_crops)]

for i, feature in enumerate(features):
    sns.boxplot(data=subset, x='label', y=feature, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{feature} by Crop', fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')

axes[-1].axis('off')
fig.suptitle('Feature Variation Across Crops (Top 6)', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Following the bootcamp ML pipeline:
1. **Label Encoding** -- Convert crop names to integers
2. **Train/Test Split** -- 80/20 stratified split
3. **Feature Scaling** -- StandardScaler (Z-score normalization)

In [ ]:
# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Feature names: {list(X.columns)}')

In [ ]:
# 1. Label Encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f'Encoded {len(label_encoder.classes_)} crop classes:')
for i, crop in enumerate(label_encoder.classes_):
    print(f'  {i:2d} -> {crop}')

In [ ]:
# 2. Train/Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# 3. Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('StandardScaler applied.')
print(f'\nBefore scaling (first sample): {X_train.iloc[0].values.round(2)}')
print(f'After scaling (first sample):  {X_train_scaled[0].round(2)}')
print(f'\nScaled mean (approx 0): {X_train_scaled.mean(axis=0).round(4)}')
print(f'Scaled std  (approx 1): {X_train_scaled.std(axis=0).round(4)}')

## 5. Model Training & Comparison

We train two different models and compare their performance:
1. **Gaussian Naive Bayes** -- Fast probabilistic baseline
2. **Random Forest Classifier** -- Powerful ensemble method with hyperparameter tuning

### 5.1 Model 1: Gaussian Naive Bayes

In [ ]:
# Train Gaussian Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

nb_preds = nb_model.predict(X_test_scaled)
nb_acc = accuracy_score(y_test, nb_preds)

print(f'Naive Bayes Accuracy: {nb_acc * 100:.2f}%')

### 5.2 Model 2: Random Forest with GridSearchCV

In [ ]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print(f'Total combinations to search: {4*4*3*3} = {4*4*3*3}')
print('\nRunning GridSearchCV with 5-fold cross validation...')

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=0,
    scoring='accuracy'
)

grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
rf_preds = best_rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_preds)

print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Random Forest Accuracy: {rf_acc * 100:.2f}%')

### 5.3 K-Fold Cross Validation

In [ ]:
# 5-Fold Cross Validation on best model
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_rf, X_train_scaled, y_train, cv=kf, scoring='accuracy')

print('K-Fold Cross Validation Results (5 Folds):')
for i, score in enumerate(cv_scores):
    print(f'  Fold {i+1}: {score*100:.2f}%')
print(f'\n  Mean: {cv_scores.mean()*100:.2f}% (+/-{cv_scores.std()*100:.2f}%)')

# Visualize CV scores
plt.figure(figsize=(8, 5))
bars = plt.bar([f'Fold {i+1}' for i in range(5)], cv_scores*100, color='#9b59b6', edgecolor='white')
plt.axhline(y=cv_scores.mean()*100, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Mean: {cv_scores.mean()*100:.2f}%')
for bar, score in zip(bars, cv_scores):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{score*100:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('K-Fold Cross Validation Scores', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Model Evaluation

Let's compare both models using classification reports and confusion matrices.

### 6.1 Model Comparison

In [ ]:
# Model Comparison
fig, ax = plt.subplots(figsize=(8, 5))
models_names = ['Gaussian\nNaive Bayes', 'Random Forest\n(GridSearchCV)']
accuracies = [nb_acc * 100, rf_acc * 100]
bar_colors = ['#3498db', '#2ecc71']

bars = ax.bar(models_names, accuracies, color=bar_colors, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylim(min(accuracies) - 2, 102)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Model Comparison -- Accuracy', fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Classification Reports

In [ ]:
print('=' * 60)
print('Classification Report -- Naive Bayes')
print('=' * 60)
print(classification_report(y_test, nb_preds, target_names=label_encoder.classes_))

In [ ]:
print('=' * 60)
print('Classification Report -- Random Forest')
print('=' * 60)
print(classification_report(y_test, rf_preds, target_names=label_encoder.classes_))

### 6.3 Confusion Matrices

In [ ]:
# Confusion Matrix -- Naive Bayes
cm_nb = confusion_matrix(y_test, nb_preds)
plt.figure(figsize=(16, 13))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
            linewidths=0.5, linecolor='white')
plt.title(f'Confusion Matrix -- Naive Bayes (Accuracy: {nb_acc*100:.1f}%)',
          fontsize=16, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix -- Random Forest
cm_rf = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(16, 13))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
            linewidths=0.5, linecolor='white')
plt.title(f'Confusion Matrix -- Random Forest (Accuracy: {rf_acc*100:.1f}%)',
          fontsize=16, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

Random Forest provides built-in feature importance -- let's see which features matter most.

In [ ]:
# Feature Importance
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

plt.figure(figsize=(10, 6))
fi_colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_names)))
sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices],
            palette=[fi_colors[i] for i in range(len(indices))])

for i, (imp, idx) in enumerate(zip(importances[indices], indices)):
    plt.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=10)

plt.title('Feature Importance in Crop Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.show()

print('\nTop 3 most important features:')
for i in range(3):
    print(f'  {i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}')

## 8. Save Model Artifacts

Save the best model, scaler, and label encoder for deployment.

In [ ]:
import os
os.makedirs('../backend/models', exist_ok=True)

# Save artifacts
best_model = best_rf if rf_acc >= nb_acc else nb_model
best_name = 'Random Forest' if rf_acc >= nb_acc else 'Naive Bayes'

joblib.dump(scaler, '../backend/models/scaler.joblib')
joblib.dump(label_encoder, '../backend/models/label_encoder.joblib')
joblib.dump(best_model, '../backend/models/best_model.joblib')

print(f'Best model ({best_name}) saved to backend/models/')
print(f'Scaler saved to backend/models/')
print(f'Label Encoder saved to backend/models/')

## 9. Conclusions

### Key Findings:

1. **Dataset Quality:** The Crop Recommendation dataset is clean (no missing values) and perfectly balanced (100 samples per crop).

2. **Model Performance:** Both Gaussian Naive Bayes and Random Forest achieved exceptional accuracy (>99%), indicating well-separated crop clusters in the feature space.

3. **Best Model:** Random Forest (with GridSearchCV tuning) was selected as the production model due to its superior generalization and robustness.

4. **Feature Importance:** Potassium (K), Humidity, and Rainfall are the most influential features for crop recommendation.

5. **Pipeline:** The complete ML pipeline (Data -> Preprocess -> Train -> Evaluate -> Deploy) was successfully implemented.

### Lessons Learned:
- Saving the **scaler and encoder alongside the model** is essential for deployment
- **Stratified splitting** ensures balanced representation in train/test sets
- Classical ML models like Random Forest can outperform complex models on tabular data

---
*End of notebook -- Model ready for deployment via Flask API*